# 12.2 · Seq2Seq 与注意力 / Sequence-to-Sequence & Attention

> **课程定位 / Where this fits**
> 第 2 课，**Part 12 · 现代 NLP 与大语言模型**。通往 Transformer 的关键一跃。
> Lesson 2, **Part 12 · Modern NLP & LLMs**. The pivotal leap toward Transformers.
>
> 很多任务是**序列→序列**：翻译(中文→英文)、摘要、问答。**Seq2Seq** 用一个 RNN **编码器**把输入压成一个**上下文向量**，再用一个 RNN **解码器**逐词生成输出。但"把整句话压成一个向量"是个**信息瓶颈**——句子越长丢得越多。**注意力机制(attention)** 的革命性想法：让解码器在生成每个词时，**直接回看输入的所有位置**并聚焦相关部分。这不仅大幅提升效果，更直接催生了 **Transformer**("Attention is all you need")。本课从零搭 Seq2Seq，**亲眼看到瓶颈导致长序列失败**，再加上注意力**完美解决**，并**可视化注意力对齐**。
> Many tasks are **sequence-to-sequence**: translation, summarization, QA. **Seq2Seq** uses an RNN **encoder** to compress input into a **context vector**, then an RNN **decoder** to generate output word by word. But "compress a whole sentence into one vector" is an **information bottleneck** — worse for long sentences. The revolutionary **attention** idea: let the decoder **look back at all input positions** and focus on relevant ones when generating each word. This boosts quality *and* directly birthed the **Transformer** ("Attention is all you need"). We build Seq2Seq from scratch, **watch the bottleneck fail on long sequences**, fix it with attention, and **visualize attention alignment**.
>
> 💼 **实战/面试视角**："seq2seq 瓶颈 / 注意力机制原理 / Bahdanau vs Luong / 注意力如何通向 Transformer" 高频。
> 💼 **Practical/interview angle:** "seq2seq bottleneck / attention mechanism / Bahdanau vs Luong / path to Transformer" — frequent.

> 📐 **符号约定 / Notation**
> - 编码器输出 $h^{enc}_1..h^{enc}_{T}$ / encoder hidden states
> - 注意力权重 $\alpha_{ij}$ —— 解码第 $i$ 步对编码第 $j$ 位的关注 / attention weight
> - 上下文向量 $c_i=\sum_j \alpha_{ij} h^{enc}_j$ / context vector

> 💡 **面试相关 / Interview-relevant**
> - "Seq2Seq 的瓶颈问题"（出镜率 ★★★★★）
> - "注意力机制的计算(score→softmax→加权求和)"（★★★★★）
> - "Bahdanau(加性) vs Luong(乘性) 注意力"（★★★）
> - "注意力为什么是通向 Transformer 的关键"（★★★★）

---

## 学习目标 / Learning Objectives
1. 理解 Seq2Seq 编码器-解码器结构。
   Understand the Seq2Seq encoder-decoder.
2. **复现"上下文向量瓶颈"**导致长序列失败。
   Reproduce the "context-vector bottleneck" failing on long sequences.
3. **从零实现注意力**并验证它解决瓶颈。
   Implement attention from scratch and verify it fixes the bottleneck.
4. **可视化注意力对齐**(看模型"看哪里")。
   Visualize attention alignment (see where the model "looks").

## 目录 / TOC
1. [Seq2Seq：编码器-解码器 ⭐](#1)
2. [瓶颈问题：长序列失败 ⭐](#2)
3. [注意力：让解码器回看输入（从零）⭐](#3)
4. [可视化注意力对齐 + 小结 ⭐](#4)


<a id="1"></a>
## 1. Seq2Seq：编码器-解码器 ⭐ / Seq2Seq: Encoder-Decoder

**Seq2Seq** 处理"输入序列→输出序列"且**两者长度可不同**(翻译时中英长度不一样)。结构两部分：
**Seq2Seq** maps "input sequence → output sequence" where **lengths can differ** (translation). Two parts:
- **编码器(encoder)**：一个 RNN 读完整个输入，把它**总结成一个上下文向量**(通常是最后的隐藏状态)。
  **Encoder:** an RNN reads the whole input and **summarizes it into a context vector** (usually the final hidden state).
- **解码器(decoder)**：另一个 RNN 从上下文向量出发，**逐词生成**输出(每步用上一个词预测下一个，自回归)。
  **Decoder:** another RNN starts from the context vector and **generates word by word** (each step predicts the next from the previous — autoregressive).

为了清楚地暴露问题，我们用一个**玩具任务：把序列反转**(输入 `[3,5,7,4]` → 输出 `[4,7,5,3]`)。它简单、可量化(整条对才算对)，而且**反转**会让注意力呈现漂亮的**反对角线**对齐，便于可视化。
To expose the problem clearly, we use a **toy task: reverse the sequence** (input `[3,5,7,4]` → output `[4,7,5,3]`). Simple, exactly scorable, and **reversal** yields a beautiful **anti-diagonal** attention pattern for visualization.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torch.nn.functional as F
sns.set_theme(style="whitegrid")
torch.manual_seed(0); np.random.seed(0)

V = 12; PAD, BOS, EOS = 0, 1, 2; DIGITS = list(range(3, V))   # 特殊符 + 数字 token / specials + digit tokens
def gen(n, L):
    src = [list(np.random.choice(DIGITS, L)) for _ in range(n)]
    tgt = [s[::-1] for s in src]                          # 目标 = 输入反转 / target = reversed input
    return src, tgt
def batchify(src, tgt):
    S = torch.tensor(src)
    T_in  = torch.tensor([[BOS] + t for t in tgt])        # 解码器输入(前面加BOS) / decoder input (prepend BOS)
    T_out = torch.tensor([t + [EOS] for t in tgt])        # 解码器目标(后面加EOS) / decoder target (append EOS)
    return S, T_in, T_out

s, t = gen(1, 6)
print(f"玩具任务: 反转序列")
print(f"  输入 src   = {s[0]}")
print(f"  目标 tgt   = {t[0]}  (= 输入反转)")
print(f"特殊符: PAD={PAD}, BOS={BOS}(解码起始), EOS={EOS}(结束)")


<a id="2"></a>
## 2. 瓶颈问题：长序列失败 ⭐ / The Bottleneck: Long Sequences Fail

先搭一个**不带注意力**的标准 Seq2Seq：编码器把整个输入压成**一个固定大小的向量**(最后隐藏状态)交给解码器。
First, a standard Seq2Seq **without attention**: the encoder compresses the entire input into **one fixed-size vector** (its final hidden state) handed to the decoder.

**直觉问题**：无论输入是 5 个词还是 50 个词，都得塞进同一个固定大小的向量。短句还行，**长句必然丢信息**——这就是**上下文向量瓶颈**。下面在 $L=6$ 和 $L=12$ 上训练，看长序列如何崩溃。
**The intuition:** whether the input is 5 or 50 words, it must fit in the same fixed vector. Fine for short inputs, but **long ones inevitably lose information** — the **context-vector bottleneck**. We train at $L=6$ and $L=12$ to watch long sequences collapse.


In [ ]:
class Encoder(nn.Module):
    def __init__(self, V, H, E=32):
        super().__init__(); self.emb = nn.Embedding(V, E); self.gru = nn.GRU(E, H, batch_first=True)
    def forward(self, x):
        out, h = self.gru(self.emb(x))                    # out: 每步隐藏状态; h: 最终隐藏状态 / all states; final
        return out, h

class Decoder(nn.Module):
    def __init__(self, V, H, E=32, use_attn=False):
        super().__init__()
        self.emb = nn.Embedding(V, E); self.gru = nn.GRU(E, H, batch_first=True); self.use_attn = use_attn
        self.out = nn.Linear(H*2 if use_attn else H, V)   # 用注意力时拼接context → 输入维度翻倍 / concat context
    def forward(self, inp, h, enc_out):
        o, h = self.gru(self.emb(inp), h)                 # 解码器 RNN / decoder RNN
        if self.use_attn:
            score = torch.bmm(o, enc_out.transpose(1, 2))     # 解码每步·编码每步 = 相关性分数 / scores (B,Td,Ts)
            attn = F.softmax(score, dim=2)                    # softmax → 注意力权重 / attention weights
            ctx = torch.bmm(attn, enc_out)                    # 按权重汇总编码器状态 = 上下文 / weighted context
            o = torch.cat([o, ctx], dim=2)                    # 拼接 / concat
            return self.out(o), h, attn
        return self.out(o), h, None

def train_seq2seq(L, use_attn, epochs=12, H=64):
    src, tgt = gen(3000, L); S, Tin, Tout = batchify(src, tgt)
    torch.manual_seed(0); enc = Encoder(V, H); dec = Decoder(V, H, use_attn=use_attn)
    opt = torch.optim.Adam(list(enc.parameters())+list(dec.parameters()), 3e-3); ce = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for i in range(0, len(S), 128):
            enc_out, h = enc(S[i:i+128])
            logits, _, _ = dec(Tin[i:i+128], h, enc_out)  # 教师强制(用真实前词) / teacher forcing
            opt.zero_grad(); ce(logits.reshape(-1, V), Tout[i:i+128].reshape(-1)).backward(); opt.step()
    sv, tv = gen(500, L); Sv, Tinv, Toutv = batchify(sv, tv)
    enc_out, h = enc(Sv); logits, _, attn = dec(Tinv, h, enc_out)
    seq_acc = (logits.argmax(2) == Toutv).all(1).float().mean().item()   # 整条序列都对才算对 / full-seq accuracy
    return seq_acc, (enc, dec)

acc = {}
for L in [6, 12]:
    acc[("no", L)], _ = train_seq2seq(L, use_attn=False)
    print(f"L={L:2}  无注意力 seq2seq: 整句准确率 = {acc[('no',L)]:.3f}")
print("\n观察: 序列变长(L=12), 固定上下文向量装不下 → 准确率崩溃(瓶颈)")


<a id="3"></a>
## 3. 注意力：让解码器回看输入（从零）⭐ / Attention: Let the Decoder Look Back

**注意力机制**直击瓶颈：不再强迫编码器把一切塞进一个向量。改为**保留编码器每个时间步的隐藏状态**，让解码器在生成每个词时，**动态地决定该关注输入的哪些位置**。三步(和 10.9/11.3 的自注意力一脉相承)：
**Attention** attacks the bottleneck head-on: instead of forcing everything into one vector, **keep all the encoder's per-step hidden states** and let the decoder **dynamically decide which input positions to focus on** when generating each word. Three steps (same idea as self-attention in 10.9/11.3):
1. **打分**：解码器当前状态与每个编码器状态算**相关性分数**(这里用点积，即 Luong 乘性注意力)。
   **Score:** the decoder's current state vs each encoder state → a **relevance score** (dot product here = Luong multiplicative attention).
2. **softmax**：把分数归一化成**注意力权重** $\alpha$(和为 1)。
   **Softmax:** normalize scores into **attention weights** $\alpha$ (sum to 1).
3. **加权求和**：用 $\alpha$ 对编码器状态加权得到**上下文向量**，喂给输出层。
   **Weighted sum:** combine encoder states by $\alpha$ into a **context vector** for the output layer.

> **Bahdanau(加性) vs Luong(乘性)**(面试)：Bahdanau 用一个小神经网络算分数(score = $v^\top\tanh(W[h_{dec};h_{enc}])$)；Luong 直接用点积(更简单更快)。本课用 Luong 点积——它正是 Transformer 自注意力的雏形。
> **Bahdanau (additive) vs Luong (multiplicative)** (interview): Bahdanau scores via a small net ($v^\top\tanh(W[h_{dec};h_{enc}])$); Luong uses a dot product (simpler/faster). We use Luong dot product — the prototype of Transformer self-attention.

下面给同一个 Seq2Seq 加上注意力(上面 `Decoder` 的 `use_attn=True`)，重新训练。
Now we add attention (the `use_attn=True` branch above) and retrain.


In [ ]:
for L in [6, 12]:
    acc[("attn", L)], models = train_seq2seq(L, use_attn=True)
    print(f"L={L:2}  带注意力 seq2seq: 整句准确率 = {acc[('attn',L)]:.3f}")

fig, ax = plt.subplots(figsize=(6.5, 4))
x = np.arange(2); w = 0.35
ax.bar(x-w/2, [acc[("no",6)], acc[("no",12)]], w, label="无注意力", color="#e67")
ax.bar(x+w/2, [acc[("attn",6)], acc[("attn",12)]], w, label="带注意力", color="#39c")
ax.set_xticks(x); ax.set_xticklabels(["L=6 (短)", "L=12 (长)"]); ax.set_ylabel("整句准确率"); ax.legend()
ax.set_title("注意力解决瓶颈: 长序列(L=12)上无注意力崩溃, 注意力近乎完美")
for xi, key in zip([x-w/2, x+w/2], ["no","attn"]):
    for j, L in enumerate([6,12]): ax.text(xi[j], acc[(key,L)]+0.01, f"{acc[(key,L)]:.2f}", ha="center", fontsize=8)
plt.tight_layout(); plt.show()
print("注意力让解码器每步直接回看输入相关位置 → 不再受单一向量瓶颈限制 → 长序列也能解")


<a id="4"></a>
## 4. 可视化注意力对齐 + 小结 ⭐ / Visualizing Attention Alignment

注意力最迷人之处：**注意力权重 $\alpha_{ij}$ 是可解释的**——它显示"生成第 $i$ 个输出词时，模型在关注第 $j$ 个输入词"。在翻译里这叫**对齐(alignment)**(英文 "cat" 对齐法文 "chat")。
The most fascinating part: **attention weights $\alpha_{ij}$ are interpretable** — they show "when generating output word $i$, the model attends to input word $j$." In translation this is **alignment** (English "cat" ↔ French "chat").

我们的任务是**反转**，所以正确的对齐应该是**反对角线**：生成第 1 个输出词时关注**最后一个**输入词，依此类推。把注意力矩阵画成热图验证。
Our task is **reversal**, so the correct alignment is **anti-diagonal**: generating the 1st output attends to the **last** input, etc. Let's plot the attention matrix as a heatmap.


In [ ]:
enc, dec = models                                         # 用 L=12 训练好的带注意力模型 / trained attn model (L=12)
sv, tv = gen(1, 12); Sv, Tinv, Toutv = batchify(sv, tv)
enc_out, h = enc(Sv)
with torch.no_grad(): logits, _, attn = dec(Tinv, h, enc_out)
A = attn[0].numpy()                                       # 注意力矩阵 (解码步 × 编码位) / (dec steps × enc positions)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(A, cmap="viridis", cbar_kws={"label":"注意力权重 α"},
            xticklabels=[str(x) for x in sv[0]], yticklabels=[str(x) for x in tv[0]]+["EOS"], ax=ax)
ax.set_xlabel("输入序列 (编码器位置)"); ax.set_ylabel("输出序列 (解码器步)")
ax.set_title("注意力对齐热图: 反对角线 = 模型学会'生成第k个输出时看倒数第k个输入'")
plt.tight_layout(); plt.show()
print("亮点沿反对角线 → 模型自己学到了反转的对齐关系(没人告诉它该对齐哪里!)")
print("这正是注意力的威力: 动态、可解释地决定'看输入的哪里'")


```
Seq2Seq: 编码器RNN把输入压成上下文向量 → 解码器RNN自回归逐词生成; 处理变长输入→变长输出
瓶颈: 整句压进一个固定向量, 长句必丢信息 → 长序列崩溃(实验: L=12 无注意力≈0)
注意力: 保留编码器每步状态; 解码每步 ①打分(点积) ②softmax得权重α ③加权求和=上下文
Bahdanau(加性,小网络打分) vs Luong(乘性,点积更简单快)
对齐可视化: α矩阵可解释(生成第i词看第j词); 反转任务→反对角线
意义: 注意力去掉瓶颈+可并行思想 → 直接催生 Transformer(把注意力用到极致, 抛弃RNN)
```

### 💡 面试速查 / Interview cheat-sheet
1. **Seq2Seq**: 编码器→上下文向量→解码器自回归生成。
   Seq2Seq: encoder → context vector → autoregressive decoder.
2. **瓶颈**: 固定向量装不下长序列信息 → 长句效果差。
   Bottleneck: a fixed vector can't hold long-sequence info.
3. **注意力三步**: 打分(点积)→softmax→对编码器状态加权求和=上下文。
   Attention: score (dot) → softmax → weighted sum of encoder states.
4. **Bahdanau vs Luong**: 加性(小网络) vs 乘性(点积)。
   Bahdanau vs Luong: additive (net) vs multiplicative (dot).
5. **通向Transformer**: 注意力可直接建模任意距离+可并行 → "Attention is all you need"。
   To Transformer: attention models any distance + parallelizable → "Attention is all you need."

### 下一节 / Next
**12.3 Transformer 从零实现**——这是全 Part 12 的核心。Transformer 把注意力用到极致：**完全抛弃 RNN**，只用**自注意力 + 前馈**，既能建模任意距离依赖、又能**完全并行**(训练快得多)。我们将**从零实现**多头自注意力、位置编码、残差+LayerNorm，搭出一个完整的 Transformer。
**12.3 Transformer from Scratch** — the heart of Part 12. Transformers take attention to the extreme: **drop RNNs entirely**, using only **self-attention + feedforward**, modeling any-distance dependencies *and* fully **parallelizable** (much faster training). We'll **implement from scratch** multi-head self-attention, positional encoding, residual+LayerNorm, building a full Transformer.
